In [ ]:
import pyspark
print(pyspark.__version__)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [ ]:
import os

# El conector Kafka es una libreria externa. Spark se comunica
# con el cluster de Kafka a traves de este conector, que traduce el flujo de
# Kafka en filas de un DataFrame de streaming. Debe coincidir con la version
# exacta de Spark y de Scala.
SPARK_VERSION = "4.0.2"
SCALA_VERSION = "2.13"

KAFKA_PKG = f"org.apache.spark:spark-sql-kafka-0-10_{SCALA_VERSION}:{SPARK_VERSION}"

# Le decimos a Spark que descargue ese paquete al arrancar.
os.environ["PYSPARK_SUBMIT_ARGS"] = f"--packages {KAFKA_PKG} pyspark-shell"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("LabKafkaColab") \
    .config("spark.jars.packages", KAFKA_PKG) \
    .getOrCreate()

print("SparkSession creada. Versión:", spark.version)

In [ ]:
# Si Spark reconoce el formato "kafka", el conector se cargo correctamente.
spark.read.format("kafka")
print("Conector Kafka disponible")

In [ ]:
# Para conectarse a un cluster, un cliente (sea productor o
# consumidor) necesita el 'bootstrap server' (host:puerto de un broker) y, si
# el cluster esta protegido, credenciales. Aiven exige autenticacion + cifrado.
import os

KAFKA_HOST = os.getenv("KAFKA_HOST", "kafka-2fe821fa-alum-139b.f.aivencloud.com:16092")
KAFKA_USER = os.getenv("KAFKA_USER", "avnadmin")
KAFKA_PASSWORD = os.getenv("KAFKA_PASSWORD", "")  # definir en el entorno, no hardcodear
TOPIC = os.getenv("KAFKA_TOPIC", "temperatura-aulas")
# Ruta al certificado CA (configurable; evita rutas hardcodeadas tipo /content/).
CA_FILE = os.getenv("KAFKA_CA", "ca.pem")

# Kafka - seguridad: SASL es el mecanismo de autenticacion (probar
# quien eres). Esta cadena 'jaas_config' (Java Authentication and Authorization Service) le entrega usuario y contrasena al
# conector usando el modulo de login SCRAM (Salted Challenge Response Authentication Mechanism), que es el que Aiven usa.
jaas_config = (
    f'org.apache.kafka.common.security.scram.ScramLoginModule required '
    f'username="{KAFKA_USER}" password="{KAFKA_PASSWORD}";'
)

In [ ]:
df_kafka = (
    spark.readStream                # readStream = leer un
                                    # flujo CONTINUO (no un archivo estatico).
    .format("kafka")                # la fuente de datos es Kafka.
    .option("kafka.bootstrap.servers", KAFKA_HOST)  # broker de entrada al cluster
    .option("subscribe", TOPIC)     # subscribe' = el
                                    # consumidor se suscribe a este topic.
    .option("startingOffsets", "latest")  # Offset: desde
                                    # que punto empezar a leer. "latest" = solo
                                    # mensajes NUEVOS (los que lleguen de ahora
                                    # en adelante). "earliest" = desde el
                                    # principio del topic, todo lo acumulado.
    .option("kafka.security.protocol", "SASL_SSL")   # autenticacion + cifrado
    .option("kafka.sasl.mechanism", "SCRAM-SHA-256") # algoritmo de autenticacion
    .option("kafka.sasl.jaas.config", jaas_config)   # credenciales
    .option("kafka.ssl.truststore.type", "PEM")      # tipo de certificado
    .option("kafka.ssl.truststore.location", CA_FILE)  # certificado CA (configurable)
    .load()
)

# Cada mensaje de Kafka tiene varios campos (key, value,
# topic, partition, offset, timestamp). El contenido util esta en 'value', y
# Kafka lo entrega como BYTES crudos. Lo convertimos a texto legible (string).
df_texto = df_kafka.selectExpr("CAST(value AS STRING) as json_str")

In [ ]:
from pyspark.sql.functions import from_json, col, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# El esquema describe la estructura de los datos. Kafka no
# sabe nada del contenido (para Kafka todo es bytes); es Spark quien le da
# sentido. Debe coincidir con lo que produce el productor.
esquema = StructType([
    StructField("aula", StringType()),
    StructField("temperatura", DoubleType()),
    StructField("humedad", DoubleType()),
    StructField("timestamp", StringType())
])

# Aunque esto se ve como transformaciones de un DataFrame
# normal, se aplican de forma CONTINUA a cada micro-lote que llega del flujo.
df_parseado = (
    df_texto
    .select(from_json(col("json_str"), esquema).alias("datos"))  # texto -> struct
    .select("datos.*")                                          # struct -> columnas
    .withColumn("timestamp", to_timestamp("timestamp"))         # texto -> fecha/hora
)

In [ ]:
query_consola = (
    df_parseado.writeStream     # writeStream define el DESTINO
                                # del flujo procesado (el 'sink').
    .format("memory")           # sink "memory": guarda los resultados en una
                                # tabla temporal en memoria, util para practicar.
    .queryName("lecturas")      # nombre de esa tabla, para consultarla con SQL.
    .outputMode("append")       # Output mode: "append" =
                                # solo agrega las filas NUEVAS de cada micro-lote.
                                # Es el modo natural para un flujo sin agregar.
    .start()                    # Arranca la consulta; corre en segundo plano.
)

In [ ]:
# La tabla "lecturas" se va llenando sola con cada micro-lote que Spark procesa.
spark.sql("SELECT * FROM lecturas ORDER BY timestamp DESC LIMIT 20").toPandas()

In [ ]:
# Spark: filter aplica la condicion a CADA fila del flujo, micro-lote
# tras micro-lote. La tabla 'alertas' solo recibira las lecturas que superen 28C.
df_alertas = df_parseado.filter(col("temperatura") > 28.0)

# Spark: esta es una SEGUNDA consulta de streaming, independiente de
# la anterior. Spark puede mantener varias consultas activas a la vez, todas
# leyendo del mismo flujo de Kafka.
query_alertas = (
    df_alertas.writeStream
    .format("memory")
    .queryName("alertas")       # nueva tabla en memoria, solo con las alertas
    .outputMode("append")
    .start()
)

In [ ]:
# La tabla 'alertas' contiene SOLO las lecturas con temperatura > 28 grados.
spark.sql("""SELECT aula, temperatura, humedad, timestamp
FROM alertas
ORDER BY timestamp DESC""").toPandas()

In [ ]:
# Spark: spark.streams.active lista todas las consultas de streaming
# que estan corriendo. Las recorremos y detenemos una por una.
for q in spark.streams.active:
    print("Deteniendo:", q.name)
    q.stop()